# 08 - Skill Taxonomy & Coverage Tracking
**Module:** `src/core/skill_selector.py`

## What this module does?
Manages the CCSS-aligned skill taxonomy and tracks which skills
have been covered across generated lessons.

## Why this matters?
Previously, the LLM invented skill strings freely, producing
inconsistent, unverifiable objectives. The taxonomy gives the
system a controlled, research-grounded skill set per grade band
and domain. The selector ensures every skill gets covered over
time before any skill is repeated.

## Research grounding
- **Rosenshine (2012)** — one measurable objective per lesson
- **Ebbinghaus (1885)** — spaced repetition: varied practice
  across different skills produces better retention than repetition
- **CCSS ELA Standards** — Speaking & Listening anchor standards
  per grade band cluster

## What we'll explore
1. The taxonomy structure — skills per band per domain
2. Skill selection — how the next skill is chosen
3. Coverage tracking — how the registry drives progression
4. The cycling behaviour when all skills are covered
5. Reading → Speaking interleaving

In [1]:
import sys
sys.path.insert(0, '..')

from src.core.skill_selector import (
    load_taxonomy,
    get_skills_for,
    get_covered_skills,
    get_next_skill,
    get_coverage_report,
)
from src.utils.file_handler import load_registry



## Part 1: The Taxonomy Structure
The full skill taxonomy lives in `data/skills/skill_taxonomy.json`.
Let's explore its structure -> grade bands, domains, and skill counts.

In [2]:
# Overview of the full taxonomy
taxonomy = load_taxonomy()

print("Taxonomy overview:\n")
for band in ["K-2", "3-5", "6-8", "9-12"]:
    domains = taxonomy[band]
    total   = sum(len(skills) for skills in domains.values())
    print(f"  {band}: {len(domains)} domains, {total} skills total")
    for domain, skills in domains.items():
        print(f"    {domain:<25} — {len(skills)} skills")
    print()

Taxonomy overview:

  K-2: 4 domains, 20 skills total
    Speaking                  — 5 skills
    Listening                 — 5 skills
    Reading                   — 5 skills
    Writing                   — 5 skills

  3-5: 4 domains, 20 skills total
    Speaking                  — 5 skills
    Listening                 — 5 skills
    Reading                   — 5 skills
    Writing                   — 5 skills

  6-8: 4 domains, 20 skills total
    Speaking                  — 5 skills
    Listening                 — 5 skills
    Reading                   — 5 skills
    Writing                   — 5 skills

  9-12: 4 domains, 20 skills total
    Speaking                  — 5 skills
    Listening                 — 5 skills
    Reading                   — 5 skills
    Writing                   — 5 skills



In [3]:
# Inspect all skills for one band + domain
print("=== K-2 Speaking Skills ===\n")
for i, skill in enumerate(taxonomy["K-2"]["Speaking"], 1):
    print(f"  {i}. {skill}")

=== K-2 Speaking Skills ===

  1. Retell a story in sequence using beginning, middle, and end
  2. Describe a person, place, or thing using sensory details
  3. Ask and answer questions about a topic using complete sentences
  4. Share an opinion and give one reason to support it
  5. Give a two-step oral instruction in the correct order


In [4]:
# Compare skill complexity across grade bands for the same domain
print("Speaking skills across grade bands:\n")
for band in ["K-2", "3-5", "6-8", "9-12"]:
    print(f"  [{band}]")
    for skill in taxonomy[band]["Speaking"]:
        print(f"    - {skill}")
    print()

Speaking skills across grade bands:

  [K-2]
    - Retell a story in sequence using beginning, middle, and end
    - Describe a person, place, or thing using sensory details
    - Ask and answer questions about a topic using complete sentences
    - Share an opinion and give one reason to support it
    - Give a two-step oral instruction in the correct order

  [3-5]
    - Present a main idea with two supporting details in logical order
    - Use transition words to connect ideas when speaking
    - Explain a process step by step in your own words
    - Respond to a peer's idea by agreeing or disagreeing with a reason
    - Adjust volume and pace to match the purpose of a presentation

  [6-8]
    - Present a claim and support it with two pieces of evidence
    - Use precise academic vocabulary appropriate to the topic
    - Acknowledge a counterclaim and explain why your position is stronger
    - Organise a spoken argument using a clear claim, evidence, and conclusion
    - Use eye c

## Part 2: Reading → Speaking Interleaving
For the `Reading → Speaking` domain, the selector interleaves
Reading and Speaking skill lists so both domains are covered evenly.

This means a student working through Reading → Speaking lessons
alternates between reading comprehension and speaking production
skills — a deliberate design choice grounded in multimodal
learning theory.

In [5]:
# See the interleaved skill list for Reading → Speaking
print("=== 6-8 Reading → Speaking (interleaved) ===\n")
interleaved = get_skills_for("6-8", "Reading → Speaking")
for i, skill in enumerate(interleaved, 1):
    source = "Reading" if i % 2 != 0 else "Speaking"
    print(f"  {i}. [{source}] {skill}")

=== 6-8 Reading → Speaking (interleaved) ===

  1. [Reading] Analyse how a central idea develops across paragraphs in a text
  2. [Speaking] Present a claim and support it with two pieces of evidence
  3. [Reading] Evaluate the strength of an author's argument using text evidence
  4. [Speaking] Use precise academic vocabulary appropriate to the topic
  5. [Reading] Determine the connotative meaning of words in an argumentative text
  6. [Speaking] Acknowledge a counterclaim and explain why your position is stronger
  7. [Reading] Compare how two authors present different perspectives on the same topic
  8. [Speaking] Organise a spoken argument using a clear claim, evidence, and conclusion
  9. [Reading] Identify the author's purpose and explain how structure supports it
  10. [Speaking] Use eye contact and deliberate pacing to engage an audience


In [6]:
# Compare with the raw Reading and Speaking lists side by side
reading  = get_skills_for("6-8", "Reading")
speaking = get_skills_for("6-8", "Speaking")

print(f"{'Reading':<55} {'Speaking'}")
print(f"{'-'*55} {'-'*55}")
for r, s in zip(reading, speaking):
    print(f"{r:<55} {s}")

Reading                                                 Speaking
------------------------------------------------------- -------------------------------------------------------
Analyse how a central idea develops across paragraphs in a text Present a claim and support it with two pieces of evidence
Evaluate the strength of an author's argument using text evidence Use precise academic vocabulary appropriate to the topic
Determine the connotative meaning of words in an argumentative text Acknowledge a counterclaim and explain why your position is stronger
Compare how two authors present different perspectives on the same topic Organise a spoken argument using a clear claim, evidence, and conclusion
Identify the author's purpose and explain how structure supports it Use eye contact and deliberate pacing to engage an audience


## Part 3: Skill Selection
`get_next_skill()` reads the registry to find which skills
have already been covered, then returns the first uncovered
skill in the taxonomy order.

Key behaviour:
- Returns the same skill on repeated calls until a lesson
  is generated and registered — it is a pure reader of state
- Only advances after `gen.generate()` writes to the registry
- When all skills are covered, cycles back to the least
  recently used skill

In [7]:
# Check next skill for each band + domain
print("Next uncovered skill per band + domain:\n")
for band in ["K-2", "3-5", "6-8", "9-12"]:
    for domain in ["Speaking", "Listening", "Reading", "Writing"]:
        skill = get_next_skill(band, domain)
        print(f"  [{band}] {domain:<25} → {skill[:55]}...")
    print()

Next uncovered skill per band + domain:

  [K-2] Speaking                  → Ask and answer questions about a topic using complete s...
  [K-2] Listening                 → Identify the main topic of a read-aloud using key detai...
  [K-2] Reading                   → Identify the main idea of a short text and one supporti...
  [K-2] Writing                   → Write a sentence that states an opinion and gives one r...

  [3-5] Speaking                  → Present a main idea with two supporting details in logi...
  [3-5] Listening                 → Summarise a speaker's main idea in one or two sentences...
  [3-5] Reading                   → Determine the meaning of a figurative expression in con...
  [3-5] Writing                   → Use transition words to connect ideas across sentences...

  [6-8] Speaking                  → Use precise academic vocabulary appropriate to the topi...
  [6-8] Listening                 → Evaluate the strength of evidence used to support a spe...
  [6-8] 

## Part 4: Coverage Reports
`get_coverage_report()` returns a full picture of progress
for a given grade band + domain.

This is what the HF Spaces UI will use to show teachers
how much of the curriculum has been covered.

In [8]:
# Full coverage report for 3-5 Speaking
report = get_coverage_report("3-5", "Speaking")

print(f"Grade Band  : {report['grade_band']}")
print(f"ELA Domain  : {report['ela_domain']}")
print(f"Total skills: {report['total']}")
print(f"Covered     : {report['covered_count']}")
print(f"Complete    : {report['complete']}")
print()

print("Covered skills:")
for skill in report['covered']:
    print(f"  ✅ {skill}")

print("\nRemaining skills:")
for skill in report['remaining']:
    print(f"  ⬜ {skill}")

Grade Band  : 3-5
ELA Domain  : Speaking
Total skills: 5
Covered     : 5
Complete    : True

Covered skills:
  ✅ Present a main idea with two supporting details in logical order
  ✅ Use transition words to connect ideas when speaking
  ✅ Explain a process step by step in your own words
  ✅ Respond to a peer's idea by agreeing or disagreeing with a reason
  ✅ Adjust volume and pace to match the purpose of a presentation

Remaining skills:


In [9]:
# Coverage snapshot across all bands and domains
print("=== FULL COVERAGE SNAPSHOT ===\n")
print(f"  {'Band':<6} {'Domain':<25} {'Progress':<12} {'Complete'}")
print(f"  {'-'*6} {'-'*25} {'-'*12} {'-'*8}")

for band in ["K-2", "3-5", "6-8", "9-12"]:
    for domain in ["Speaking", "Listening", "Reading", "Writing"]:
        r = get_coverage_report(band, domain)
        progress = f"{r['covered_count']}/{r['total']}"
        complete = "✅" if r['complete'] else "⬜"
        print(f"  {band:<6} {domain:<25} {progress:<12} {complete}")
    print()

=== FULL COVERAGE SNAPSHOT ===

  Band   Domain                    Progress     Complete
  ------ ------------------------- ------------ --------
  K-2    Speaking                  2/5          ⬜
  K-2    Listening                 0/5          ⬜
  K-2    Reading                   0/5          ⬜
  K-2    Writing                   0/5          ⬜

  3-5    Speaking                  5/5          ✅
  3-5    Listening                 0/5          ⬜
  3-5    Reading                   2/5          ⬜
  3-5    Writing                   1/5          ⬜

  6-8    Speaking                  1/5          ⬜
  6-8    Listening                 0/5          ⬜
  6-8    Reading                   0/5          ⬜
  6-8    Writing                   0/5          ⬜

  9-12   Speaking                  0/5          ⬜
  9-12   Listening                 0/5          ⬜
  9-12   Reading                   0/5          ⬜
  9-12   Writing                   0/5          ⬜



## Part 5: The Cycling Behaviour
Once all skills in a band + domain are covered, the selector
cycles back to the least recently used skill — the one that
was covered earliest.

This ensures continued variety even after full coverage,
while respecting the order in which skills were originally taught.

In a production system, cycling would be replaced by a
personalised selection based on individual student performance —
prioritising skills where the student scored lowest.

In [12]:
from src.core.generator import LessonGenerator
from src.core.skill_selector import get_coverage_report

gen = LessonGenerator(verbose=False)
req = [("3-5", "Speaking", "The environment")]

for grade, domain, theme in req:
    lesson = gen.generate(grade_band=grade, ela_domain=domain, theme=theme)
    print(f"✅ {lesson['lesson_id']} | {lesson['metadata']['primary_skill'][:60]}")

print()
report = get_coverage_report("3-5", "Speaking")
print(f"Coverage: {report['covered_count']}/{report['total']}")
print(f"Covered  : {report['covered']}")
print(f"Remaining: {report['remaining']}")

[checks] All pre-generation checks passed ✅
[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Use transition words to connect ideas when speaking'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within 3-5 vocabulary ceiling (60 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band '3-5'.
[checks] cultural_bias_check: ✅ [PASS] No cultural bias terms detected.
[validator] Step 4 — Guardrail checks complete ✅
[validator] Validation passed for lesson: L-35-SPK-009
[file_handler] Saved lesson → D:\projects\bantrly-lesson-generator\data\generated\L-35-SPK-020.json
[file_handler] Registered combo → 3-5 | The environment | Use transition words to connect ideas when speaking
✅ L-35-SPK-020 | Use transition words to connect ideas when speaking

Coverage: 5/5

In [15]:
# Simulate full coverage by checking a band that has lessons generated
# Then confirm the cycling behaviour kicks in

report = get_coverage_report("3-5", "Speaking")

if report['complete']:
    next_skill = get_next_skill("3-5", "Speaking")
    print("All skills covered — cycling behaviour active.")
    print(f"Next skill (least recently used): {next_skill}")
    print(f"\nThis skill was first covered by lesson generated at:")
    registry = load_registry()
    first = next(e for e in registry["used_combinations"] 
                if e["skill"] == next_skill)
    print(f"  {first['lesson_id']} | {first['generated_at']}")
else:
    covered = report['covered_count']
    total   = report['total']
    remaining = report['remaining']
    print(f"Coverage not yet complete ({covered}/{total} skills covered).")
    print(f"Generate {total - covered} more lesson(s) to trigger cycling.")
    print(f"\nNext up: {remaining[0]}")



All skills covered — cycling behaviour active.
Next skill (least recently used): Explain a process step by step in your own words

This skill was first covered by lesson generated at:
  L-35-SPK-012 | 2026-03-16T17:16:01.283959


In [14]:
from src.core.skill_selector import get_covered_skills
covered = get_covered_skills("3-5", "Speaking")
print(f"len(covered): {len(covered)}")
print(f"cycle_index : {len(covered) % 5}")
print()
for i, s in enumerate(covered):
    print(f"  {i}: {s}")

len(covered): 12
cycle_index : 2

  0: Present a main idea with two supporting details in logical order
  1: Use transition words to connect ideas when speaking
  2: Explain a process step by step
  3: Explain a process step by step in your own words
  4: Respond to a peer's idea by agreeing or disagreeing with a reason
  5: Adjust volume and pace to match the purpose of a presentation
  6: Present a main idea with two supporting details in logical order
  7: Present a main idea with two supporting details in logical order
  8: Present a main idea with two supporting details in logical order
  9: Present a main idea with two supporting details in logical order
  10: Present a main idea with two supporting details in logical order
  11: Use transition words to connect ideas when speaking


## Summary

| What we explored | Key takeaway |
|---|---|
| Taxonomy structure | 4 bands × 4 domains × 5 skills = 80 skills total |
| Reading → Speaking | Interleaves Reading and Speaking lists for even coverage |
| `get_next_skill()` | Pure registry reader — only advances after a lesson is registered |
| `get_coverage_report()` | Full progress snapshot per band + domain |
| Coverage snapshot | Shows which bands and domains have lessons generated |
| Cycling behaviour | After full coverage, cycles to least recently used skill |

**Design decision this validates:**
Skills are stored as plain strings in a JSON file — not as
database records or enums. This means adding a new skill to the
taxonomy requires editing one JSON file, with no code changes.
A teacher or curriculum designer could maintain the taxonomy
without touching Python.

**Research connection:**
The ordered progression through the taxonomy directly implements
Rosenshine's (2012) principle of presenting material in small steps
with a clear, measurable objective at each step. The cycling
behaviour implements Ebbinghaus's (1885) spaced repetition —
skills are revisited after other skills have been practiced,
not immediately repeated.

**Known limitation:**
Skill selection is currently FIFO within the taxonomy order.
A production system would weight selection by:
- Individual student performance on each skill
- Time since last practice (spaced repetition interval)
- Difficulty relative to the student's current level